# Laboratorio 01: Implementación de Clustering (K-Means y Jerárquico)

**Curso:** Inteligencia Artificial  
**Fecha:** 10 Febrero 2026

En este laboratorio implementaremos desde cero el algoritmo K-Means y lo compararemos con soluciones de librería (`scikit-learn`). También exploraremos el Clustering Jerárquico y aplicaremos estos métodos a diversos datasets (Iris, Pingüinos, Vinos, Países) y problemas (cuantización de color).

### Librerías utilizadas

En este laboratorio se utilizarán las siguientes librerías:

- **NumPy**: Para manejo eficiente de arreglos numéricos y operaciones matemáticas básicas. Es fundamental para la implementación manual de K-Means.
- **Matplotlib**: Para la visualización de datos, incluyendo gráficos de dispersión (scatter plots) y dendrogramas.
- **Pandas**: Para la carga y manipulación de conjuntos de datos tabulares (DataFrames), como los datos de vinos o Iris.
- **Seaborn**: Para cargar datasets de ejemplo (como 'penguins') y mejorar la estética de los gráficos.
- **Scikit-learn (sklearn)**: 
    - `datasets`: Para cargar conjuntos de datos estándar (Iris, make_moons).
    - `cluster`: Para utilizar las implementaciones optimizadas de `KMeans` y `AgglomerativeClustering` (jerárquico) y compararlas con nuestra implementación manual.
- **SciPy**: 
    - `cluster.hierarchy`: Específicamente para calcular y graficar los dendrogramas del agrupamiento jerárquico.
- **Pillow (PIL)**: Para cargar y manipular imágenes en el ejercicio de cuantización de color.

> **Nota:** > En el inciso 1, el algoritmo de *k-means* se implementa **desde cero**, utilizando únicamente NumPy. Las funciones de `scikit-learn` se utilizan posteriormente solo para efectos de comparación y en los incisos avanzados.

In [11]:
# Instalación de librerías necesarias (Descomentar si es necesario)
# !pip install numpy pandas matplotlib seaborn scikit-learn scipy pillow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Módulos de Scikit-learn
from sklearn.datasets import load_iris, make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering

# Módulos de SciPy para dendrogramas
from scipy.cluster.hierarchy import dendrogram, linkage

# Configuración de gráficos
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style="whitegrid")

### 1.
 Implementación de K-means

Programar desde cero (sin usar librerías de datos ni funciones especializadas), un método de agrupamiento k-means, para vectores en $\mathbb{R}^{d}$.

Como **input** se deberá dar a la función:
* La matriz de datos, una matriz numérica de $n \times d$.
* El número de grupos o clústers a construir.

Como **output**, su algoritmo debe producir:
* Un vector con las clases o labels de cada dato (vector de tamaño $n$).
* Un vector con los centroides de cada grupo (matriz de $k \times d$ donde cada fila de esta matriz es un centroide).

In [12]:
def kmeans_manual(X, k, max_iters=100, tol=1e-4):
    """
    Implementación de K-Means desde cero para vectores en R^d.
    
    Parámetros:
    - X: Matriz de datos (n x d). 'n' muestras, 'd' características.
    - k: Número de grupos (clústers).
    - max_iters: Máximo de iteraciones si no converge antes.
    
    Retorna:
    - labels: Vector (n) con la clase asignada a cada dato.
    - centroids: Matriz (k x d) con los centros de cada grupo.
    """
    
   
    X = np.array(X)
    n_samples, n_features = X.shape
    

    random_indices = np.random.choice(n_samples, k, replace=False)
    centroids = X[random_indices]
    
    labels = np.zeros(n_samples)
    

    for i in range(max_iters):

        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        

        new_labels = np.argmin(distances, axis=1)
        

        new_centroids = np.zeros((k, n_features))
        for cluster_idx in range(k):
  
            points_in_cluster = X[new_labels == cluster_idx]
            
            if len(points_in_cluster) > 0:
      
                new_centroids[cluster_idx] = points_in_cluster.mean(axis=0)
            else:
              
                new_centroids[cluster_idx] = centroids[cluster_idx]
        


        if np.allclose(centroids, new_centroids, atol=tol):
            break
            
        centroids = new_centroids
        labels = new_labels

    return labels, centroids

### 2.
 Evaluar el funcionamiento de su algoritmo con los siguientes conjuntos de datos:

a) El conjunto de datos Iris:  
from sklearn.datasets import load_iris  
import pandas as pd  

iris = load_iris()  
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)  
df['target'] = iris.target  

b) El conjunto de datos penguins de la librería  
import seaborn as sns  
penguins = sns.load_dataset("penguins")

c) El conjunto de datos winequality-red, dentro del conjunto de datos wines:  
https://archive.ics.uci.edu/dataset/186/wine+quality  

En cada caso, elegir un número apropiado de clústers para el agrupamiento.  
Contrastar los resultados de su algoritmo de k-means contra los de la librería scikit-learn,  
y discutir las semejanzas o diferencias.


### a) Conjunto de datos Iris

El conjunto de datos Iris contiene 150 observaciones con 4 características.
Dado que existen tres clases naturales, se elige k = 3.


In [13]:
# Cargar Iris
iris = load_iris()
df_iris = pd.DataFrame(data=iris.data, columns=iris.feature_names)

X_iris = df_iris.values

# K-means manual
labels_iris_manual, centroids_iris_manual = kmeans_manual(X_iris, k=3)

# K-means scikit-learn
kmeans_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_iris_sklearn = kmeans_iris.fit_predict(X_iris)


### b) Conjunto de datos Penguins

El conjunto de datos Penguins contiene información morfológica de distintas especies.
Se eliminan valores faltantes y se consideran únicamente variables numéricas.
Dado que existen tres especies, se elige k = 3.


In [14]:
# Cargar dataset Penguins
penguins = sns.load_dataset("penguins")

# Seleccionar variables numéricas y eliminar NaNs
penguins_numeric = penguins.select_dtypes(include="number").dropna()
X_penguins = penguins_numeric.values

# K-means manual
labels_penguins_manual, centroids_penguins_manual = kmeans_manual(X_penguins, k=3)

# K-means scikit-learn
kmeans_penguins = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_penguins_sklearn = kmeans_penguins.fit_predict(X_penguins)


### c) Conjunto de datos Wine Quality (winequality-red)

Este conjunto de datos contiene características fisicoquímicas del vino tinto.
Se elige k = 6, considerando los distintos niveles de calidad.


In [15]:
# Cargar dataset Wine Quality Red
wine = pd.read_csv("winequality-red.csv", sep=";")
X_wine = wine.values

# K-means manual
labels_wine_manual, centroids_wine_manual = kmeans_manual(X_wine, k=6)

# K-means scikit-learn
kmeans_wine = KMeans(n_clusters=6, random_state=42, n_init=10)
labels_wine_sklearn = kmeans_wine.fit_predict(X_wine)


En el conjunto de datos Iris se conoce que existen tres clases naturales,
correspondientes a las tres especies de flores presentes en el dataset.
Por este motivo se elige k = 3, con el objetivo de evaluar si el algoritmo
k-means es capaz de recuperar dicha estructura sin utilizar las etiquetas.


### 3. 
Hacer un agrupamiento jerárquico con los datos de los países countries_binary.xlsx.

Visualizar los resultados o dendrogramas de diferentes métodos de agrupamiento,
variando los siguientes:

- el método de agrupamiento (simple, completo, promedio, Ward)
- la métrica utilizada (euclideana, Hamming).


In [ ]:
# 1. Cargar el dataset
try:
    file_path = "countries_binary.xlsx"
    df_countries = pd.read_excel(file_path)
    
    # Asumimos que la primera columna son los nombres de los países
    # y el resto son las características binarias
    country_names = df_countries.iloc[:, 0].values
    X_countries = df_countries.iloc[:, 1:].values
    
    print(f"Dataset cargado: {X_countries.shape[0]} países con {X_countries.shape[1]} características.")

except FileNotFoundError:
    print(f"Error: El archivo {file_path} no se encuentra.")
    # Datos dummy por si falla la carga para demostrar el código
    country_names = [f"País {i}" for i in range(10)]
    X_countries = np.random.randint(0, 2, (10, 5))

# 2. Configuración de métodos y métricas
# Nota: 'ward' solo funciona correctamente con distancia euclideana en scipy
configs = [
    ('single', 'euclidean'),
    ('complete', 'euclidean'),
    ('average', 'euclidean'),
    ('ward', 'euclidean'),
    ('single', 'hamming'),
    ('complete', 'hamming'),
    ('average', 'hamming')
]

# 3. Generar Dendrogramas
plt.figure(figsize=(20, 24))

for i, (method, metric) in enumerate(configs, 1):
    plt.subplot(4, 2, i)
    
    # Calcular la matriz de enlace (linkage matrix)
    try:
        Z = linkage(X_countries, method=method, metric=metric)
        
        # Graficar dendrograma
        dendrogram(
            Z,
            labels=country_names,
            leaf_rotation=90,
            leaf_font_size=8
        )
        
        plt.title(f"Agrupamiento Jerárquico\nMétodo: {method} - Métrica: {metric}")
        plt.xlabel("Países")
        plt.ylabel("Distancia")
        
    except Exception as e:
        plt.text(0.5, 0.5, f"Error en config: {method}/{metric}\n{e}", 
                 ha='center', va='center')
        plt.title(f"Método: {method} - Métrica: {metric}")

plt.tight_layout()
plt.show()

### 4. 
Realizar un análisis de agrupamiento k-means, nuevamente para los datos de los países,
que están disponibles en el archivo countries_binary.xlsx.

Contrastar con el ejercicio anterior. ¿Son iguales las agrupaciones? ¿Por qué?
Justificar.


In [ ]:
# 1. Aplicar K-Means con un k comparable 
# (Observando los dendrogramas previos, podemos elegir un corte o un k arbitrario razonable)
k_countries = 6
kmeans = KMeans(n_clusters=k_countries, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_countries)

# 2. Mostrar los grupos formados
print(f"Resultados de K-Means (k={k_countries}):")
df_results = pd.DataFrame({'País': country_names, 'Cluster': labels_kmeans})

for i in range(k_countries):
    paises_cluster = df_results[df_results['Cluster'] == i]['País'].tolist()
    print(f"\nCluster {i}:")
    print(", ".join(paises_cluster))

### --- ANÁLISIS COMPARATIVO ---

¿Son iguales las agrupaciones?
En general, NO suelen ser idénticas. Es probable ver diferencias significativas entre los grupos de K-means y los del agrupamiento jerárquico (especialmente el basado en métrica Hamming).

¿Por qué? Justificación:

1. Naturaleza de los Datos y Métricas de Distancia:
   - K-means utiliza distancias Euclidianas y calcula 'centroides' promediando las posiciones de los puntos. 
   - El dataset es BINARIO (0s y 1s). El promedio de vectores binarios da valores decimales (ej. 0.5) que no existen en el espacio original de las características.
   - La distancia Euclideana no siempre es la mejor para datos categóricos/binarios. La distancia de Hamming (usada en una de las variantes jerárquicas) es más natural para este tipo de datos, ya que cuenta el número de desacuerdos bit a bit.

2. Forma de los Clusters:
   - K-means asume clusters esféricos (o hiperesféricos) y de tamaño similar debido a su optimización basada en varianza.
   - El agrupamiento jerárquico no asume una forma geométrica específica en los clusters; simplemente agrupa basado en proximidad directa (linkage), permitiendo formas más arbitrarias o 'alargadas' si la conectividad lo dicta.

3. Estructura del Algoritmo:
   - K-means es un algoritmo de partición plana (flat clustering).
   - El jerárquico construye una estructura anidada (árbol), donde las decisiones tomadas en pasos tempranos (agrupaciones iniciales) son irrevocables.

### 5.
 Generar un conjunto de datos sintéticos, de tamaño 200, con make_moons o círculos concéntricos de Scikit-learn. (Aquí la idea es realizar dos clúster,
con 100 observaciones cada una).

Una vez tenga fijado el conjunto de datos, comparar los algoritmos de clustering.


In [ ]:
# 1. Generar datos sintéticos (Lunas)
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=42)

# 2. Aplicar K-Means
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_moons)

# 3. Aplicar Agrupamiento Jerárquico
hierarchical = AgglomerativeClustering(n_clusters=2, linkage='single', metric='euclidean')
labels_hierarchical = hierarchical.fit_predict(X_moons)

# 4. Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot K-Means
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_kmeans, cmap='viridis', edgecolor='k')
axes[0].set_title("K-Means (Falla en formas no convexas)")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")

# Plot Jerárquico
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_hierarchical, cmap='viridis', edgecolor='k')
axes[1].set_title("Jerárquico (Enlace Simple / Single Linkage)")
axes[1].set_xlabel("Feature 1")

plt.tight_layout()
plt.show()

#### (a) ¿Por qué k-means falla con formas de "luna" o anillos concéntricos? 

K-means asume que los clústers son convexos y esféricos (isotrópicos). Funciona minimizando la varianza dentro del clúster basándose en la distancia euclidiana a un centroide central. En formas alargadas, curvas o "lunas", el centroide geométrico no representa bien la forma, y el algoritmo parte la forma por la mitad con una línea recta (frontera de decisión lineal).

#### (b) Comparación K-means vs. Jerárquico:

Mejor resultado: El agrupamiento jerárquico produce mejores resultados en este caso.

Método y métrica ideal: El método de Enlace Simple (Single Linkage) con métrica euclidiana funciona mejor. Esto se debe a que el enlace simple agrupa basándose en la distancia mínima entre puntos de dos clústers, lo que permite seguir la "cadena" de datos (efecto de encadenamiento) y adaptarse a formas no convexas como las lunas.

### 6.
 Realizar un algoritmo de cuantización de colores para imágenes RGB,
usando como base un algoritmo de agrupamiento.

Ilustrar los resultados obtenidos con 3 imágenes de su elección. En cada una mostrar:

- El mapa de clases resultado del agrupamiento.
- La imagen cuantizada resultante.


In [ ]:
def cuantizar_imagen(image_path, n_colors=8):
    """
    Función para cuantizar los colores de una imagen usando K-Means.
    """
    try:
        # 1. Cargar imagen
        original_img = Image.open(image_path)
        original_img = np.array(original_img)
        
        # Normalizar a 0-1 si es necesario (KMeans funciona bien con floats)
        img_float = original_img / 255.0
        
        # 2. Transformar imagen a matriz de datos (N píxeles x 3 canales RGB)
        w, h, d = original_img.shape
        image_array = np.reshape(img_float, (w * h, d))
        
        # 3. Ajustar K-Means
        kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=5)
        kmeans.fit(image_array)
        
        # 4. Predecir las etiquetas para cada píxel
        labels = kmeans.predict(image_array)
        
        # 5. Reconstruir la imagen cuantizada
        quantized_img_array = kmeans.cluster_centers_[labels]
        quantized_img = np.reshape(quantized_img_array, (w, h, d))
        
        # 6. Mapa de clases (labels)
        labels_img = np.reshape(labels, (w, h))
        
        return original_img, labels_img, quantized_img
        
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {image_path}")
        return None, None, None

mis_imagenes = ["imagen1.png", "imagen2.png", "imagen3.png"] 

for img_name in mis_imagenes:
    orig, labels, quant = cuantizar_imagen(img_name, n_colors=8)
    
    if orig is not None:
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        
        axs[0].imshow(orig)
        axs[0].set_title("Imagen Original")
        axs[0].axis('off')
        
        axs[1].imshow(labels, cmap='tab20')
        axs[1].set_title("Mapa de Clases (Clusters)")
        axs[1].axis('off')
        
        axs[2].imshow(quant)
        axs[2].set_title(f"Cuantizada (k=8 colores)")
        axs[2].axis('off')
        
        plt.show()